# Kestrel V2 — Roost consolidation, fixed and verified

Runs entirely from the public GitHub repo and the **Kestrel V1** release assets — no Google Drive, no sign-in.

1. Download V1 weights + the Roost benchmark data from the v1.0.0 release.
2. `scripts.roost_v2`: 50-fact benchmark under a settings sweep (run 0 = exact V1 settings under the corrected test), then — if a setting passes the gate — teach V1 true facts about itself → **Kestrel-Nano V2**.
3. Export V2, check the C# engine against PyTorch, and download a small results bundle (results, parity report, the changed memory tables).

`Runtime → Run all`. ~30–40 min on an A100.

In [ ]:
# 0. GPU
import torch, os, subprocess
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(torch.__version__, '|', name)
assert torch.cuda.is_available(), 'needs a GPU runtime (A100 preferred)'

In [ ]:
# 1. Code + V1 assets (public: no credentials)
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers
REL = 'https://github.com/YoMosa2009/Kestrel/releases/download/v1.0.0'
A = '/content/assets'; os.makedirs(A, exist_ok=True)
for f in ['kestrel-nano-v1-weights.pt', 'roost-data.zip']:
    if not os.path.exists(f'{A}/{f}'):
        !wget -q -O {A}/{f} {REL}/{f}
    print(f, os.path.getsize(f'{A}/{f}') // 1_000_000, 'MB')
if not os.path.isdir(f'{A}/roost-data'):
    !unzip -q -o {A}/roost-data.zip -d {A}/roost-data
print(sorted(os.listdir(f'{A}/roost-data')))

In [ ]:
# 2. Benchmark sweep + build V2 (log is kept in /content/v2/log.txt)
OUT = '/content/v2'; os.makedirs(OUT, exist_ok=True)
# The first V2 sweep (settings 0-2, Oct 6) showed lr alone is not enough: answer NLL 6.66 -> 6.31 at
# lr 1e-2; answer-only loss helped but was still slow, which exposed a slot-selection bug
# (only the LAST loop's memory reads were trainable). This run: fixed selection + answer-only loss.
# Set ONLY = '' to run the whole sweep.
ONLY = 'fix-ans-lr1e-2,fix-ans-lr3e-2,fix-ans-lr1e-1'
only = f'--only {ONLY}' if ONLY else ''
!python -u -m scripts.roost_v2 --ckpt {A}/kestrel-nano-v1-weights.pt --replay-dir {A}/roost-data --out {OUT} {only} 2>&1 | tee {OUT}/log.txt
import json
R = json.load(open(f'{OUT}/results.json'))
for r in R['benchmark']:
    print(f"{r['name']:<18} recall {r['recall_before']:.0%}->{r['recall_after']:.0%}  qa {r['qa_before']:.0%}->{r['qa_after']:.0%}  "
          f"nll {r['nll_before']:.2f}->{r['nll_after']:.2f}  reg {r['regression_pct']:+.2f}%  {'PASS' if r['accepted'] else 'fail'}")
print('best:', R['best'], '| benchmark passed:', R['benchmark_passed'], '| V2 built:', R['v2_built'])
print('V1 chat:', R['v1_chat']); print('V2 chat:', R.get('v2_chat'))

In [ ]:
# 3. Export V2 and check the native C# engine against PyTorch
if R['v2_built']:
    W = f'{OUT}/kestrel-nano-v2-weights.pt'
    !python -m scripts.export_gguf --ckpt {W} --out /content/kestrel-nano-v2-f32.gguf --dtype f32 --name "Kestrel-Nano V2" --version V2 --fixture {OUT}/parity-v2.json
    !python -m scripts.export_gguf --ckpt {W} --out /content/kestrel-nano-v2-f16.gguf --dtype f16 --name "Kestrel-Nano V2" --version V2
    !md5sum /content/kestrel-nano-v2-f16.gguf | tee {OUT}/v2-f16.md5
    DOTNET = os.path.expanduser('~/.dotnet/dotnet')
    if not os.path.exists(DOTNET):
        !curl -sSL https://dot.net/v1/dotnet-install.sh | bash -s -- --channel 8.0 > /dev/null
    !DOTNET_CLI_TELEMETRY_OPTOUT=1 {DOTNET} run -c Release --project app/Kestrel.Cli -- parity /content/kestrel-nano-v2-f32.gguf {OUT}/parity-v2.json | tee {OUT}/parity-v2-f32.txt
    !DOTNET_CLI_TELEMETRY_OPTOUT=1 {DOTNET} run -c Release --project app/Kestrel.Cli -- parity /content/kestrel-nano-v2-f16.gguf {OUT}/parity-v2.json | tee {OUT}/parity-v2-f16.txt
else:
    print('V2 not built - the benchmark did not pass the gate; results are still downloaded.')

In [ ]:
# 4. Download the results bundle (results, log, parity, and the V2 memory tables - the only weights V2 changes)
import zipfile
from google.colab import files
Z = '/content/kestrel-v2-results.zip'
with zipfile.ZipFile(Z, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        if f != 'kestrel-nano-v2-weights.pt':
            z.write(f'{OUT}/{f}', f)
print(Z, os.path.getsize(Z) // 1_000_000, 'MB')
files.download(Z)